# Create Permanent Email — No CAPTCHA, No Phone
Uses **mail.tm** public API — account is permanent (not disposable), inbox accessible via web and API.

In [ ]:
import requests, json, random, string

BASE = 'https://api.mail.tm'

# Step 1: Get available domains
r = requests.get(f'{BASE}/domains', timeout=10)
domains = r.json()['hydra:member']
domain = domains[0]['domain']
print(f'✅ Available domain: {domain}')

# Step 2: Create account
suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
EMAIL = f'penux.research.{suffix}@{domain}'
PASSWORD = 'PenuX2026!Research'

r = requests.post(f'{BASE}/accounts', json={'address': EMAIL, 'password': PASSWORD}, timeout=10)
print(f'Account creation: {r.status_code}')
if r.status_code in (200, 201):
    print(f'✅ Email created: {EMAIL}')
    print(f'   Password: {PASSWORD}')
    print(f'   Inbox: https://mail.tm (login with above credentials)')
else:
    print(f'❌ Error: {r.text}')

# Step 3: Get JWT token (to read emails later)
r = requests.post(f'{BASE}/token', json={'address': EMAIL, 'password': PASSWORD}, timeout=10)
if r.status_code == 200:
    TOKEN = r.json()['token']
    print(f'✅ Token obtained (save this for reading inbox later)')
else:
    TOKEN = None
    print(f'Token: {r.status_code}')

print(f'\n📧 YOUR NEW PERMANENT EMAIL: {EMAIL}')
print(f'🔑 PASSWORD: {PASSWORD}')
print(f'📥 INBOX: https://mail.tm')

In [ ]:
# Read inbox (run after registration emails arrive)
if TOKEN:
    headers = {'Authorization': f'Bearer {TOKEN}'}
    r = requests.get(f'{BASE}/messages', headers=headers, timeout=10)
    msgs = r.json().get('hydra:member', [])
    print(f'Inbox: {len(msgs)} messages')
    for m in msgs:
        print(f"  From: {m['from']['address']} | Subject: {m['subject']}")
        # Get full message to find verification link
        r2 = requests.get(f"{BASE}/messages/{m['id']}", headers=headers, timeout=10)
        body = r2.json().get('text', '') or r2.json().get('html', '')
        # Find links
        import re
        links = re.findall(r'https?://[^\s<>"]+(?:confirm|verif|activ)[^\s<>"]*', body)
        if links:
            print(f'  🔗 Verification link: {links[0]}')
else:
    print('No token — run cell above first')